# LUMEN-299 · 2 — Designing the mirror

**The failure.** Michelson's Annapolis apparatus of 1879 gave 299,944 km/s. Newcomb built
a larger copy in Washington three years later and got 299,860. The two disagreed by
84 km/s, about one and a half combined standard errors, and both men took the obvious
next step: more readings. Neither number moved.

The information a reading carries about *c* is a property of the apparatus, not of
patience, and so is the one thing no number of readings can separate. This notebook writes
the rotating mirror down as an expression tree and asks the design questions in the order
they constrain each other: what a reading is worth, what it can never tell apart, and what
the only fix costs.

| | reaches into |
|---|---|
| the apparatus as one tree | `core.ModelSpec`, `core.Apply`/`Div`, `Mirror.forward` |
| what a reading is worth, by baseline | `design.fisher_information`, `design.expected_posterior_sd` |
| what readings cannot separate | `design.identifiability_ridge`, `design.estimable_combinations`, `design.observation_from_model` |
| the survey that separates it | `Observation`, `design.expected_posterior_sd` with a survey prior |
| Annapolis against Washington | `design.power_from_se`, `design.eig_gaussian` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import pandas as pd

import lightspeed as L

from axiom.core import Unsupported
from axiom.design import (
    EstimabilityReport, FisherInformation, IdentifiabilityRidge, eig_gaussian,
    estimable_combinations, expected_posterior_sd, fisher_information, identifiability_ridge,
    observation_from_model, power_from_se,
)

from axiom.display import enable, table

sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, caption, curve_band, lines, mark_x, mark_y

enable();  # every axiom result renders itself from here on

model, mirror = L.model(), L.surface()
model

## 1 · One tree, five apparatus

The mean is the residual image displacement, $\delta = 8\pi\,\omega D / c - 4\pi/n$: the
mirror turns $2\pi\omega \cdot 2D/c$ during the round trip, a reflection doubles it, and an
$n$-faced mirror tuned to its next face subtracts $4\pi/n$. Two parameters, both
logarithms so the information matrix is scale-free; two data columns, the rotation rate
and the null angle. Everything else in the case study reads this tree.

In [ ]:
rows = []
for e in L.HISTORY:
    if e.family != "rotating-mirror":
        continue
    data = L.apparatus(e.omega, e.faces)
    delta = float(mirror.forward(data, L.point(e.baseline_km))[0])
    full = delta + (4 * math.pi / e.faces if e.faces else 0.0)
    sd = float(L.angular_noise(e.baseline_km))
    per_reading = L.C_TRUE * sd / full
    rows.append([f"{e.year} {e.who}", f"{e.baseline_km:,.3f}", f"{e.omega:.0f}",
                 str(e.faces or "plane"), f"{full:.4f}", f"{full * 10_000:.1f}",
                 f"{sd:.2e}", f"{per_reading:,.0f}"])
table(rows, headers=("apparatus", "D (km)", "rev/s", "faces", "deflection (rad)",
                     "mm at a 10 m lever", "sd per reading (rad)", "c per reading (km/s)"),
      title="what each rotating mirror reads, through one forward()")

check = L.point(35.385)
data = L.apparatus(528.0, 8)
print("\nthe inverse reads the tree backwards:",
      f"{float(mirror.speed(data, mirror.forward(data, check), 35.385)[0]):,.3f} km/s from a "
      f"deflection generated at {math.exp(check['log_c']):,.3f}")

The last column is the whole design problem. A reading at Foucault's twenty metres is
worth three thousand kilometres a second; one at Mount Wilson is worth forty. The
column is not the geometry alone — $\delta$ grows with $D$, but so does the blur of an
image that has crossed seventy kilometres of air — and the design functions are what
turn that trade into a number.

## 2 · What a reading is worth, by baseline

`fisher_information` differentiates `forward` with respect to every parameter and divides
by the noise. Do it at one reading per baseline, from twenty metres to a hundred
kilometres, with the noise the module says a reading at that baseline has.

In [ ]:
baselines = np.geomspace(0.02, 100.0, 60)
per_reading = []
for D in baselines:
    faces = 8 if D > 5.0 else None
    omega = L.null_omega(D, faces) if faces else min(L.OMEGA_MAX, 257.0)
    info = fisher_information(mirror, L.apparatus(omega, faces), L.point(D),
                              float(L.angular_noise(D)), method="finite")
    assert isinstance(info, FisherInformation)
    i = info.index("log_c")
    per_reading.append(L.C_TRUE / math.sqrt(info.as_array()[i, i]))
per_reading = np.asarray(per_reading)
geometry = per_reading[0] * (baselines[0] / baselines)  # if the noise did not grow at all

fig = lines(
    baselines, {"with the blur the air adds": per_reading, "if the image did not dim": geometry},
    colors=(BLUE, ORANGE),
    title="What one reading is worth, by baseline",
    subtitle="the standard error one reading puts on c, km/s — the mirror spun as fast as a tuning fork allows",
    x_title="baseline D (km)", y_title="c per reading (km/s)",
)
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")
mark_x(fig, 0.605, text="Annapolis 1879")
mark_x(fig, 35.385, text="Mt Wilson 1926", right=True)
caption(fig, "Geometry alone promises a reading worth 1/D. The dimming of the return "
             "takes part of that back: past a kilometre the standard error falls like "
             "1/sqrt(D), so the mountain is worth about sixteen Annapolis readings per "
             "reading rather than the sixty the geometry advertises.")
at = lambda D: float(np.interp(math.log(D), np.log(baselines), per_reading))
print(f"per reading: {at(0.605):,.0f} km/s at Annapolis, {at(35.385):,.0f} at Mount Wilson "
      f"({at(0.605) / at(35.385):.1f}x); geometry alone would have said {35.385 / 0.605:.0f}x")

## 3 · The thing readings cannot separate

A speed is a distance over a time. The tree contains $D$ and $c$ only as the ratio $D/c$,
and `identifiability_ridge` says so before any reading is taken: the flattest direction of
the information matrix is *raise $c$ and lengthen $D$ together*, along which nothing the
mirror sees changes, and the matrix is singular. `estimable_combinations` names the one
combination a season of readings does estimate.

In [ ]:
season = L.apparatus(L.null_omega(35.385, 8), 8, n=1600)
ridge = identifiability_ridge(mirror, season, L.point(35.385), float(L.angular_noise(35.385)),
                              method="finite")
assert isinstance(ridge, IdentifiabilityRidge)
print("1,600 readings at Mount Wilson, no survey:")
print("  condition number:", ridge.condition_number)
print("  flattest direction:", {k: round(v, 3) for k, v in ridge.direction.items()})

readings = observation_from_model(model, season, noise_sd=float(L.angular_noise(35.385)),
                                  label="1600 readings")
alone = estimable_combinations([readings], L.point(35.385), parameters=("log_c", "log_D"),
                               scaling="absolute")
assert isinstance(alone, EstimabilityReport)
print(f"  rank {alone.rank}/2; flat direction: {[c.label for c in alone.symmetries]}")

with_survey = estimable_combinations(
    [readings, L.survey_observation(35.385, L.MOUNT_WILSON.relative_error)],
    L.point(35.385), parameters=("log_c", "log_D"), scaling="absolute")
assert isinstance(with_survey, EstimabilityReport)
print(f"  ...plus one survey of the baseline: rank {with_survey.rank}/2; "
      f"estimable: {[c.label for c in with_survey.estimable]}")

So the apparatus needs a second observation, of $D$ alone, and how good that observation
is decides what the season is worth. Treat the survey as the prior on $\log D$ and read
`expected_posterior_sd` for $\log c$ at three grades of survey — paced, taped, and
triangulated by the Coast and Geodetic Survey — at both sites.

In [ ]:
surveys = {"paced (1%)": 1e-2, "steel tape (1 in 10^5)": L.ANNAPOLIS.relative_error,
           "geodetic triangulation (1 in 7 million)": L.MOUNT_WILSON.relative_error}
sites = {"Annapolis 1879, 100 readings": (0.605, 257.0, None, 100),
         "Mount Wilson 1926, 1,600 readings": (35.385, L.null_omega(35.385, 8), 8, 1600)}
rows = []
for site, (D, omega, faces, n) in sites.items():
    info = fisher_information(mirror, L.apparatus(omega, faces, n=n), L.point(D),
                              float(L.angular_noise(D)), method="finite")
    assert isinstance(info, FisherInformation)
    i = info.index("log_c")
    statistical = L.C_TRUE / math.sqrt(info.as_array()[i, i])
    cells = [site, f"{statistical:,.1f}"]
    for name, rel in surveys.items():
        sds = expected_posterior_sd({"log_c": 1.0, "log_D": rel}, info)
        assert isinstance(sds, dict)
        cells.append(f"{L.C_TRUE * sds['log_c']:,.1f}")
    rows.append(cells)
table(rows, headers=("season", "readings alone (if D were known)", *surveys),
      title="what the season is worth, in km/s on c, by how well the baseline is known")

Read the Annapolis row. Its hundred readings are worth 60 km/s on their own; a paced
baseline turns that into 3,000, and a steel tape leaves it at 61 — the tape is good enough
and the readings are the limit. At Mount Wilson the readings are worth one kilometre a
second and **the steel tape would have been the limit**: a tape at one part in a hundred
thousand puts three kilometres a second on $c$ whatever the mirror does. The triangulation
at one part in seven million is what let 1,600 readings mean something, and it is not a
statistical fact — it is a survey party on a mountain for a season.

## 4 · Annapolis against Washington

The 1879 and 1882 results differ by 84 km/s. Could readings have settled it? Price the
disagreement at both apparatus with `power_from_se`, then price what each season *did*
buy against the prior of the day with `eig_gaussian`.

In [ ]:
def se_after(D, omega, faces, n):
    info = fisher_information(mirror, L.apparatus(omega, faces, n=n), L.point(D),
                              float(L.angular_noise(D)), method="finite")
    assert isinstance(info, FisherInformation)
    i = info.index("log_c")
    return L.C_TRUE / math.sqrt(info.as_array()[i, i])

DISAGREEMENT = 299_944.0 - 299_860.0
rows = []
for n in (100, 400, 1600, 6400):
    se_79 = se_after(0.605, 257.0, None, n)
    se_82 = se_after(3.721, 230.0, None, n)
    combined = math.sqrt(se_79**2 + se_82**2)
    rows.append([n, f"{se_79:,.0f}", f"{se_82:,.0f}",
                 f"{power_from_se(DISAGREEMENT, combined).power:.2f}"])
table(rows, headers=("readings at each", "se 1879 (km/s)", "se 1882 (km/s)",
                     "power to see 84 km/s"),
      title="could more readings have told Annapolis from Washington?")

PRIOR_SD = 300.0  # what the 1870s believed c to: Cornu's ±200 and the disagreement with Foucault
rows = []
for label, (D, omega, faces, n) in {"Annapolis 1879 (100)": (0.605, 257.0, None, 100),
                                    "Washington 1882 (100)": (3.721, 230.0, None, 100),
                                    "Mount Wilson 1926 (1,600)": (35.385, L.null_omega(35.385, 8), 8, 1600)}.items():
    se = se_after(D, omega, faces, n)
    rows.append([label, f"{se:,.1f}", f"{eig_gaussian(PRIOR_SD, se):.2f}"])
table(rows, headers=("season", "se on c (km/s)", "nats against a ±300 prior"),
      title="what each season bought")

Sixteen hundred readings at each site would have had the power — four hundred gives
seven chances in ten — and in the world of notebook 1 the disagreement was not in the
readings: the two apparatus carry a *shared*
systematic of the Annapolis generation and one of their own each, and the survey and the
tuning fork are where those live. Readings that resolve a difference to 84 km/s resolve
it to the wrong answer with a narrower interval, which is the shape of every failure in
this case study.

The nats column is the honest ranking of the three seasons: Mount Wilson bought close to
six nats against the prior of the 1870s where Annapolis bought one and a half and
Washington two and a half, and it bought them with a baseline — a survey, a road up a
mountain, and a season — rather than with readings.

## What notebook 2 established

1. **A reading's worth is a property of the baseline**, and the dimming of the return
   takes part of it back: the standard error per reading falls like $1/\sqrt{D}$ past a
   kilometre rather than $1/D$. Mount Wilson is worth sixteen Annapolis readings per
   reading, not sixty.
2. **$c$ and $D$ are one number to the mirror.** `identifiability_ridge` reports a
   singular matrix whose flat direction is *both together*, and
   `estimable_combinations` names $\log D - \log c$ as the only thing a season estimates.
3. **The survey is the anchor**, and its grade is the design decision. A steel tape would
   have capped Mount Wilson at 3 km/s; the geodetic triangulation is what let the readings
   count.
4. **Readings could not have settled 1879 against 1882**, because the disagreement was
   not in them. Notebook 3 runs the 1926 season and files what it finds.